# Vehicle Routing Problem (VRP)

Find a set of routes of minimal total length for a fleet of vehicles visiting a set of locations, starting from a central depot.

Modeled with a `Circuit` constraint and **dummy depots** (one per vehicle). Following the circuit, each stretch between two depot copies is one vehicle's tour. Depot-to-depot arcs cost 0, so unused vehicles can stay at the depot for free.

If there is only one vehicle, the problem reduces to the TSP.


In [1]:
import cpmpy as cp
import numpy as np


## Distance matrix

Planar Euclidean distances between coordinates.


In [2]:
def compute_euclidean_distance_matrix(locations):
    """Computes distances between all points."""
    locs = np.asarray(locations)
    return np.linalg.norm(locs[:, None, :] - locs[None, :, :], axis=-1).astype(int)


## VRP model

Prepend `n_vehicles` copies of the depot to the customer locations, then post a single `Circuit` over the expanded node set.


In [3]:
def vrp(locations, depot, n_vehicles, demand=None, capacity=None):
    """Build a Circuit-based VRP by prepending `n_vehicles` copies of `depot`
    to `locations` (customers).

    Optional `demand` (one value per customer) and `capacity` add capacity
    constraints.

    Returns (model, (succ,)) or (model, (succ, load)) when capacity is set.
    """
    locs = [depot] * n_vehicles + list(locations)
    n = len(locs)
    depots = list(range(n_vehicles))
    customers = list(range(n_vehicles, n))

    distance_matrix = cp.cpm_array(compute_euclidean_distance_matrix(locs))

    # succ[i]=j means j is visited immediately after i
    succ = cp.intvar(0, n - 1, shape=n)
    model = cp.Model(cp.Circuit(succ))
    model.minimize(cp.sum(distance_matrix[i, succ[i]] for i in range(n)))

    if demand is not None and capacity is not None:
        demand_n = [0] * n_vehicles + list(demand)
        load = cp.intvar(0, capacity, shape=n)
        model += [load[d] == 0 for d in depots]  # leave depot empty
        model += [(succ[i] == j).implies(load[j] == load[i] + demand_n[j])
                  for i in range(n) for j in customers]
        return model, (succ, load)

    return model, (succ,)


## Instance data

Toy instance with planar coordinates (depot + 8 customers, capacity constraints).


In [4]:
depot = (288, 149)
locations = [  # customers
    (288, 129), (270, 133), (256, 141), (256, 163),
    (236, 169), (228, 169), (228, 148), (220, 164),
]
demand = [3, 4, 8, 8, 10, 5, 3, 9]  # per customer
n_vehicles, capacity = 3, 20


## Solve


In [5]:
model, (succ, load) = vrp(locations, depot, n_vehicles=n_vehicles,
                          demand=demand, capacity=capacity)

# number of vehicles used (depot-to-depot arc = unused)
depots = list(range(n_vehicles))
num_used = n_vehicles - cp.sum(cp.InDomain(succ[d], depots) for d in depots)

D = compute_euclidean_distance_matrix([depot] * n_vehicles + locations)

if model.solve():
    print(model.status())
    print("Total Cost of solution", int(model.objective_value()))
    print("Vehicles used:", num_used.value())

    s = succ.value()
    vehicle = 0
    for d in depots:
        if s[d] in depots:
            continue  # unused
        stops = ["0 (load 0)"]
        dist = 0
        i = d
        while s[i] not in depots:
            j = s[i]
            dist += D[i, j]
            cust = j - n_vehicles + 1
            stops.append(f"{cust} (load {load[j].value()})")
            i = j
        dist += D[i, s[i]]
        stops.append("0")
        print(f"Vehicle {vehicle}: {' --> '.join(stops)}  (km={dist})")
        vehicle += 1
else:
    raise ValueError("Model is unsatisfiable")


ExitStatus.OPTIMAL (0.084333 seconds)
Total Cost of solution 344
Vehicles used: 3
Vehicle 0: 0 (load 0) --> 1 (load 3) --> 2 (load 7) --> 3 (load 15) --> 0  (km=86)
Vehicle 1: 0 (load 0) --> 4 (load 8) --> 5 (load 18) --> 0  (km=109)
Vehicle 2: 0 (load 0) --> 6 (load 5) --> 8 (load 14) --> 7 (load 17) --> 0  (km=149)
